In [ ]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
import os

from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model,AgentState

load_dotenv(override=True)

DEEPSEEK_API = os.getenv("DEEPSEEK_API")

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=DEEPSEEK_API,
    # other params...
)

from langchain.messages import SystemMessage, HumanMessage, AIMessage
system_message = "你叫小智，是一名乐于助人的智能助手。请在对话中保持温和，有耐心的语气。"
system_message = SystemMessage(content=system_message)

messages = [
    system_message
]

from typing import Any
from langgraph.runtime import Runtime
@before_model
def check_limit(state: AgentState, runtime: Runtime) -> dict[str, Any]:
    print("请求模型：")
    if len(state["messages"]) > 2:
        return {
            "messages": [AIMessage(content="对话轮数过多，请重新开始对话。")],
            "jump_to": "end"
        }
    return None

@after_model
def log_response(state: AgentState, runtime: Runtime) -> None:
    print("模型响应：", state["messages"][-1].content)
    return None


agent = create_agent(
    model=model,
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。",
    middleware=[
        check_limit,
        log_response,
    ],
)

messages.append(HumanMessage(content="我叫陈明"))
result = agent.invoke({
    "messages": messages,
})
print(result)
messages.append(AIMessage(content=result["messages"][-1].content))
messages.append(HumanMessage(content="我20岁"))
result = agent.invoke({
    "messages": messages,
})
print(result)
messages.append(AIMessage(content=result["messages"][-1].content))
messages.append(HumanMessage(content="我希望你叫小天"))
result = agent.invoke({
    "messages": messages,
})
print(result)
messages.append(AIMessage(content=result["messages"][-1].content))
messages.append(HumanMessage(content="你叫什么"))
result = agent.invoke({
    "messages": messages,
})
print(result)


模型响应： 陈明你好！很高兴认识你！有什么我可以帮助你的吗？
{'messages': [SystemMessage(content='你叫小智，是一名乐于助人的智能助手。请在对话中保持温和，有耐心的语气。', additional_kwargs={}, response_metadata={}, id='82fd1f24-d2de-4b14-bf6c-9064df1d8cf1'), HumanMessage(content='我叫陈明', additional_kwargs={}, response_metadata={}, id='d916bda7-2066-44f8-8508-648b6a5324e1'), AIMessage(content='陈明你好！很高兴认识你！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 46, 'total_tokens': 60, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 46}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '43deac4f-18ed-49b3-b351-186cc7690f89', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b86d1-95dc-74a0-b19a-7fbc3f5030e3-0', usage_metadata={'input_tokens': 46, 'output_tokens': 14, 'total_tokens': 6